In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch

In [15]:
mm = pd.read_excel('/Users/taliacho/Downloads/Ranger/Bryan-Ranger/body_comp/US_data_1.29.24.xlsx')
mm['BM'] = mm[['B1M', 'B2M', 'B3M']].mean(axis=1,skipna=True)
mm['BA'] = mm[['B1A', 'B2A', 'B3A']].mean(axis=1,skipna=True)
mm['AA'] = mm[['A1A', 'A2A', 'A3A']].mean(axis=1,skipna=True)
mm['AM'] = mm[['A1M', 'A2M', 'A3M']].mean(axis=1,skipna=True)
mm['QA'] = mm[['Q1A', 'Q2A', 'Q3A']].mean(axis=1,skipna=True)
mm['QM'] = mm[['Q1M', 'Q2M', 'Q3M']].mean(axis=1,skipna=True)

mm_avg = mm[['Study ID','BM','BA','AM','AA','QM','QA']]

In [16]:
hp = pd.read_excel('/Users/taliacho/Downloads/Ranger/Bryan-Ranger/body_comp/Data_11.5.23.xlsx')
hp_avg = hp[['FM', 'FFM']]

df = pd.concat([mm_avg, hp_avg], axis=1)

In [17]:
def split_data(data, train_size=0.8, test_size=0.1, random_state=None):
    train, temp = train_test_split(data, train_size=train_size, random_state=random_state)
    test_and_valid_size = 1.0 - train_size
    test_ratio = test_size / test_and_valid_size
    test, valid = train_test_split(temp, test_size=test_ratio, random_state=random_state)
    return train, test, valid

In [21]:
mse = np.zeros(12)
rmse = np.zeros(12)
mae = np.zeros(12)
mape = np.zeros(12)

In [26]:
for index in range(12):
    train, test, valid = split_data(torch.tensor(df.iloc[:, [index, 7, 8]].values, dtype=torch.float32))
    
    # Remove rows with NaN values
    train = train[~torch.isnan(train).any(axis=1)]
    test = test[~torch.isnan(test).any(axis=1)]
    
    train_x = train[:, 0].reshape(-1, 1)  # Reshape for regression
    train_y = train[:, 1]
    test_x = test[:, 0].reshape(-1, 1)
    test_y = test[:, 1]

    # Linear Regression Model
    model = LinearRegression()
    model.fit(train_x.numpy(), train_y.numpy())  # Convert tensors to numpy arrays for fitting

    # Predictions
    preds = model.predict(test_x.numpy())  # Convert tensor to numpy array

    # Calculate Metrics
    mse[index] = mean_squared_error(test_y.numpy(), preds)  # Convert tensor to numpy array
    rmse[index] = np.sqrt(mse[index])
    mae[index] = mean_absolute_error(test_y.numpy(), preds)  # Convert tensor to numpy array
    mape[index] = np.mean(np.abs((preds - test_y.numpy()) / test_y.numpy())) * 100  # Convert tensor to numpy array

IndexError: positional indexers are out-of-bounds

In [ ]:
body_parts = ["BM FM", "BM FFM", "BA FM", "BA FFM", "AM FM",
              "AM FFM", "AA FM", "AA FFM", "QM FM", "QM FFM",
              "QA FM", "QA FFM"]
metrics = ["MAE", "MSE", "RMSE", "MAPE"]

header = "{:<10} {:<10} {:<10} {:<10} {:<10}".format("Body Part", *metrics)
print(header)
print("-" * len(header))

for index, part in enumerate(body_parts):
    rmse_value = rmse[index].item()  
    mse_value = mse[index].item()  
    mae_value = mae[index].item()  
    mape_value = mape[index].item()  
    row = "{:<10} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(part, mae_value, mse_value, rmse_value, mape_value)
    print(row)

In [ ]:
rmse_np = rmse.detach().cpu()
mae_np = mae.detach().cpu()

positions = torch.arange(len(body_parts))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(positions - width / 2, rmse_np.flatten(), width, label='RMSE', color='#1f77c4')
rects2 = ax.bar(positions + width / 2, mae_np.flatten(), width, label='MAE', color='#9f79b4')

ax.set_ylabel('%')
ax.set_title('RMSE and MAE by Body Part')
ax.set_xticks(positions)
ax.set_xticklabels(body_parts)
ax.legend()

plt.xticks(rotation=45)
plt.show()